<a href="https://colab.research.google.com/github/kady05naik/LeetCode/blob/main/2026_08_28.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.master('local').appName('myapp').getOrCreate()

**1.Write a solution to find the second highest distinct salary from the Employee table.**

If there is no second highest salary, return null
```
Example 1:
Employee table:
+----+--------+
| id | salary |
+----+--------+
| 1  | 100    |
| 2  | 200    |
| 3  | 300    |
+----+--------+
Output:
+---------------------+
| SecondHighestSalary |
+---------------------+
| 200                 |
+---------------------+
```



```
SELECT max(salary) AS SecondHighestSalary
FROM employee
WHERE salary < (SELECT MAX(salary) FROM employee);
```



In [8]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

data = [(1, 100), (2, 200), (3, 300)]
schema=("id Integer,salary Integer")
raw=spark.createDataFrame(data,schema)

WindowSpec=Window.orderBy(F.col('salary').desc())
df=raw.withColumn('rnk', F.dense_rank().over(WindowSpec))
df.select(F.col("salary").alias('SecondHighestSalary')).filter("rnk=2").show()

+-------------------+
|SecondHighestSalary|
+-------------------+
|                200|
+-------------------+



**2. Write a solution to find the nth highest distinct salary from the Employee table. If there are less than n distinct salaries, return null.**

The result format is in the following example.
```
Input:
Employee table:
+----+--------+
| id | salary |
+----+--------+
| 1  | 100    |
| 2  | 200    |
| 3  | 300    |
+----+--------+
n = 2
Output:
+------------------------+
| getNthHighestSalary(2) |
+------------------------+
| 200                    |
+------------------------+
```



```
CREATE FUNCTION getNthHighestSalary(N INT) RETURNS INT
BEGIN
  RETURN (
    SELECT Distinct salary
    FROM (
        SELECT
            id,
            salary,
            DENSE_RANK() OVER(ORDER BY salary DESC) AS rnk
        FROM
            employee
    )t
    WHERE rnk=n
 );
```



In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("EmployeeDataFrame").getOrCreate()

schema = StructType([
    StructField("Id", IntegerType(), True),
    StructField("Salary", IntegerType(), True)
])
data = [
    (1, 100),
    (2, 200),
    (3, 300)
]
employee_df = spark.createDataFrame(data, schema=schema)


